# Solver Full-Trajectory LoRA SFT on Colab T4

This notebook trains `Qwen/Qwen2.5-0.5B-Instruct` with LoRA on the full-trajectory solver SFT data.

Before running:

1. In Colab, choose `Runtime > Change runtime type > T4 GPU`.
2. Make sure this notebook is inside the `new_solver` folder, with `train_lora_sft.py`, `prepare_full_trajectory_sft.py`, and `requirements.txt` next to it.
3. Make sure `../normalized_outputs/solver_full_trajectory_dataset.jsonl` exists, or upload/copy it into that relative location.
4. Have a Hugging Face token ready if you want to log in or avoid anonymous download limits.


In [10]:
# Confirm GPU availability.
!nvidia-smi


Sun May  3 03:09:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Mount Google Drive

Run this if your repo or dataset is stored in Google Drive.

In [11]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!ls
%cd drive/MyDrive/Final\ Project
%cd new_solver/
!ls


shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
ls: cannot open directory '.': Transport endpoint is not connected


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_42827/279571259.py", line 2, in <cell line: 0>
    get_ipython().run_line_magic('cd', 'drive/MyDrive/Final\\ Project')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurre

Exception ignored in atexit callback: <bound method InteractiveShell.atexit_operations of <google.colab._shell.Shell object at 0x7d2a233a42f0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3924, in atexit_operations
    self.reset(new_session=False)
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 1445, in reset
    self.history_manager.reset(new_session)
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_history.py", line 33, in reset
    super(ColabHistoryManager, self).reset(new_session=new_session)
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/history.py", line 592, in reset
    self.dir_hist[:] = [os.getcwd()]
                        ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected


## Install Dependencies

In [ ]:
!pip install -q -r requirements.txt


## Hugging Face Login

Hugging Face login is optional for Qwen/Qwen2.5-0.5B-Instruct, but useful if you hit rate limits or use private models.

In [ ]:
from huggingface_hub import notebook_login
# notebook_login()

## Prepare SFT Splits

This reads `../normalized_outputs/solver_full_trajectory_dataset.jsonl` and writes `data/train_sft.jsonl`, `data/val_sft.jsonl`, and `data/test_sft.jsonl`. If the files already exist, this cell still safely regenerates them.

In [ ]:
from pathlib import Path

!pwd
!ls -lh ../normalized_outputs/solver_full_trajectory_dataset.jsonl



input_jsonl = Path('../normalized_outputs/solver_full_trajectory_dataset.jsonl')
if not input_jsonl.exists():
    raise FileNotFoundError(
        f'Missing {input_jsonl.resolve()}. Upload or copy normalized_outputs next to new_solver.'
    )

!python prepare_full_trajectory_sft.py \
  --input ../normalized_outputs/solver_full_trajectory_dataset.jsonl \
  --output-dir data


In [ ]:
# Quick split sanity check.
import json
from collections import Counter
from pathlib import Path

for path in [Path('data/train_sft.jsonl'), Path('data/val_sft.jsonl'), Path('data/test_sft.jsonl')]:
    counts = Counter()
    final_answer_rows = 0
    marker_rows = 0
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            counts[row['source']] += 1
            final_answer_rows += bool(row.get('final_answer'))
            marker_rows += 'Final Answer:' in row['messages'][-1]['content']
    print(path, sum(counts.values()), dict(counts), 'final_answer_rows=', final_answer_rows, 'marker_rows=', marker_rows)


## Train LoRA Adapter

These settings are conservative for a T4. If you hit CUDA out-of-memory, set `MAX_SEQ_LENGTH = 1024` and rerun this cell.

In [ ]:
MAX_SEQ_LENGTH = 2048
EPOCHS = 3
OUTPUT_DIR = 'outputs/qwen2.5-0.5b-solver-lora'

!python train_lora_sft.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --train-file data/train_sft.jsonl \
  --val-file data/val_sft.jsonl \
  --output-dir {OUTPUT_DIR} \
  --max-seq-length {MAX_SEQ_LENGTH} \
  --epochs {EPOCHS} \
  --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 1 \
  --gradient-accumulation-steps 16 \
  --learning-rate 2e-4 \
  --lora-r 16 \
  --lora-alpha 32 \
  --lora-dropout 0.05 \
  --fp16


## Save Adapter

In [ ]:
# Zip the LoRA adapter for download.
!zip -r qwen2.5-0.5b-solver-lora.zip outputs/qwen2.5-0.5b-solver-lora

from google.colab import files
files.download('qwen2.5-0.5b-solver-lora.zip')


In [ ]:
# Optional: copy the adapter to Google Drive instead of downloading it.
from google.colab import drive

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/new_solver_outputs
!cp -r outputs/qwen2.5-0.5b-solver-lora /content/drive/MyDrive/new_solver_outputs/
